# Flash Attention: Algorithm, Memory Access Patterns & Benchmarks

**Key insight from the module**: LLM inference is memory-bandwidth bound. Standard attention materializes the full N×N attention matrix in HBM, causing O(N²) memory reads/writes. Flash Attention computes attention in SRAM tiles — never materializing the full matrix — reducing HBM access from O(N²) to O(N).

This notebook:
1. Explains the Flash Attention algorithm (tiling + online softmax)
2. Compares memory access patterns: standard vs flash
3. Benchmarks PyTorch's scaled_dot_product_attention with/without flash backend
4. Measures scaling behavior across sequence lengths

In [ ]:
import sys
sys.path.insert(0, '../../..')

import torch
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
import numpy as np
from utils import benchmark, gpu_info

gpu = gpu_info.detect_gpu()
print(f"GPU: {gpu.name} | VRAM: {gpu.vram_gb:.1f} GB | BW: {gpu.bw_gbs} GB/s | FP16: {gpu.tflops_fp16} TFLOPS")
print(f"Ridge point: {gpu.ridge_point:.1f} FLOP/byte")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

## 1. The Flash Attention Algorithm

### Why standard attention is memory-inefficient

Standard attention computes:


For sequence length N=4096, head_dim=128:
- S matrix: 4096² × 2 bytes = **32 MB per head**
- P matrix: another **32 MB per head**
- With 32 heads: **2 GB** just for intermediate attention matrices

### Flash Attention's tiling strategy

Flash Attention splits Q, K, V into blocks that fit in SRAM (~20 MB on A100):
1. Load a block of Q (Br × d) into SRAM
2. Iterate over blocks of K, V (Bc × d)
3. Compute local attention scores, track running max for numerically stable softmax
4. Accumulate output using online softmax correction
5. Write final output block to HBM — **never write S or P to HBM**

### Memory access comparison

| Operation | Standard Attention | Flash Attention |
|-----------|-------------------|-----------------|
| HBM reads | O(Nd + N²) | O(Nd × N²/M) where M = SRAM size |
| HBM writes | O(Nd + N²) | O(Nd) |
| Peak memory | O(N²) | O(N) |
| IO complexity | Θ(Nd + N²) | Θ(N²d²/M) |

For typical M >> d², Flash Attention is **IO-optimal** — it achieves the minimum possible HBM accesses.

In [ ]:
def memory_access_standard_attention(seq_len, head_dim, num_heads, dtype_bytes=2):
    """Calculate HBM bytes read/written for standard attention."""
    N, d, h = seq_len, head_dim, num_heads
    # Read Q, K, V from HBM
    reads = 3 * N * d * dtype_bytes * h
    # Write S = QK^T to HBM, read it back for softmax
    reads += N * N * dtype_bytes * h  # read S for softmax
    writes = N * N * dtype_bytes * h  # write S
    # Write P (after softmax) to HBM, read for P@V
    reads += N * N * dtype_bytes * h  # read P
    writes += N * N * dtype_bytes * h  # write P
    # Write output
    writes += N * d * dtype_bytes * h
    return reads + writes

def memory_access_flash_attention(seq_len, head_dim, num_heads, sram_bytes=20*1024*1024, dtype_bytes=2):
    """Calculate HBM bytes read/written for flash attention."""
    N, d, h = seq_len, head_dim, num_heads
    M = sram_bytes
    # Block sizes (simplified): Bc = M / (4*d*dtype), Br = min(M/(4*d*dtype), d)
    Bc = min(N, max(1, M // (4 * d * dtype_bytes)))
    Br = min(N, max(1, min(Bc, d)))
    num_blocks_k = (N + Bc - 1) // Bc
    num_blocks_q = (N + Br - 1) // Br
    # Each Q block loaded once per K block pass; K,V loaded once per Q block
    reads = h * (num_blocks_q * Br * d * dtype_bytes * num_blocks_k +  # Q reloads
                 num_blocks_k * Bc * d * dtype_bytes * 2 * num_blocks_q)  # K,V
    # Output written once
    writes = h * N * d * dtype_bytes
    return reads + writes

seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
standard_bytes = [memory_access_standard_attention(s, 128, 32) / 1e9 for s in seq_lens]
flash_bytes = [memory_access_flash_attention(s, 128, 32) / 1e9 for s in seq_lens]

plt.figure(figsize=(10, 5))
plt.semilogy(seq_lens, standard_bytes, 'ro-', linewidth=2, label='Standard Attention')
plt.semilogy(seq_lens, flash_bytes, 'bs-', linewidth=2, label='Flash Attention')
plt.xlabel('Sequence Length')
plt.ylabel('Total HBM Access (GB)')
plt.title('HBM Memory Access: Standard vs Flash Attention')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('memory_access_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"At seq_len=4096: Standard={standard_bytes[5]:.2f} GB, Flash={flash_bytes[5]:.2f} GB")
print(f"Ratio: {standard_bytes[5]/flash_bytes[5]:.1f}x fewer HBM accesses with Flash")

## 2. Peak Memory Usage

The critical difference: standard attention allocates O(N²) memory for the attention matrix, while Flash Attention only needs O(N) for the output plus small SRAM buffers.

In [ ]:
def measure_peak_memory(fn, *args):
    """Measure peak GPU memory allocated during fn execution."""
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    before = torch.cuda.memory_allocated()
    fn(*args)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    return (peak - before) / 1024**2  # MB

def standard_attention(q, k, v):
    scale = q.shape[-1] ** -0.5
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    attn = torch.softmax(scores, dim=-1)
    return torch.matmul(attn, v)

def flash_attention(q, k, v):
    return F.scaled_dot_product_attention(q, k, v, enable_gqa=False)

batch, heads, head_dim = 1, 32, 128
test_seqs = [512, 1024, 2048, 4096]
standard_mem, flash_mem = [], []

for seq_len in test_seqs:
    q = torch.randn(batch, heads, seq_len, head_dim, dtype=torch.float16, device='cuda')
    k = torch.randn(batch, heads, seq_len, head_dim, dtype=torch.float16, device='cuda')
    v = torch.randn(batch, heads, seq_len, head_dim, dtype=torch.float16, device='cuda')
    
    std_mb = measure_peak_memory(standard_attention, q, k, v)
    flash_mb = measure_peak_memory(flash_attention, q, k, v)
    standard_mem.append(std_mb)
    flash_mem.append(flash_mb)
    print(f"seq_len={seq_len:5d} | Standard: {std_mb:8.1f} MB | Flash: {flash_mb:8.1f} MB | Savings: {std_mb-flash_mb:.1f} MB")
    del q, k, v
    torch.cuda.empty_cache()

plt.figure(figsize=(8, 5))
x = np.arange(len(test_seqs))
plt.bar(x - 0.2, standard_mem, 0.4, label='Standard', color='#ef4444')
plt.bar(x + 0.2, flash_mem, 0.4, label='Flash', color='#3b82f6')
plt.xticks(x, test_seqs)
plt.xlabel('Sequence Length')
plt.ylabel('Peak Memory (MB)')
plt.title('Peak GPU Memory: Standard vs Flash Attention')
plt.legend()
plt.tight_layout()
plt.savefig('peak_memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. PyTorch SDPA Backends

 dispatches to different backends:
- **Flash Attention** (flash_sdp): Tiled, IO-aware — fastest for long sequences
- **Memory-Efficient** (mem_efficient_sdp): xFormers-based — good fallback
- **Math** (math_sdp): Naive implementation — materializes full attention matrix

We can force specific backends using context managers.

In [ ]:
# Check which backends are available
print("SDPA Backend Availability:")
print(f"  Flash SDP:           {torch.backends.cuda.flash_sdp_enabled()}")
print(f"  Memory-efficient SDP: {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"  Math SDP:            {torch.backends.cuda.math_sdp_enabled()}")

# Verify flash attention works
q = torch.randn(1, 8, 256, 64, dtype=torch.float16, device='cuda')
k = torch.randn(1, 8, 256, 64, dtype=torch.float16, device='cuda')
v = torch.randn(1, 8, 256, 64, dtype=torch.float16, device='cuda')

with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
    try:
        out = F.scaled_dot_product_attention(q, k, v)
        print("
✅ Flash Attention backend works!")
    except RuntimeError as e:
        print(f"
⚠️  Flash Attention not available: {e}")
        print("   Falling back to mem_efficient backend for benchmarks")
del q, k, v
torch.cuda.empty_cache()

## 4. Benchmarking: Flash vs Math Backend

We benchmark  forcing each backend separately, measuring latency across sequence lengths.

In [ ]:
def bench_sdpa(seq_len, batch=1, heads=32, head_dim=128, backend='flash', warmup=5, iters=20):
    """Benchmark SDPA with a specific backend."""
    q = torch.randn(batch, heads, seq_len, head_dim, dtype=torch.float16, device='cuda')
    k = torch.randn(batch, heads, seq_len, head_dim, dtype=torch.float16, device='cuda')
    v = torch.randn(batch, heads, seq_len, head_dim, dtype=torch.float16, device='cuda')
    
    if backend == 'flash':
        ctx = torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False)
    elif backend == 'math':
        ctx = torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=False)
    elif backend == 'mem_efficient':
        ctx = torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True)
    else:
        raise ValueError(f"Unknown backend: {backend}")
    
    with ctx:
        # Warmup
        for _ in range(warmup):
            F.scaled_dot_product_attention(q, k, v)
        torch.cuda.synchronize()
        
        # Timed iterations
        start = time.perf_counter()
        for _ in range(iters):
            F.scaled_dot_product_attention(q, k, v)
        torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) / iters * 1000
    
    del q, k, v
    torch.cuda.empty_cache()
    return elapsed_ms

# Benchmark across sequence lengths
seq_lens = [256, 512, 1024, 2048, 4096, 8192]
flash_times, math_times = [], []

for s in seq_lens:
    try:
        ft = bench_sdpa(s, backend='flash')
    except RuntimeError:
        ft = bench_sdpa(s, backend='mem_efficient')
    flash_times.append(ft)
    
    try:
        mt = bench_sdpa(s, backend='math')
    except torch.cuda.OutOfMemoryError:
        mt = float('nan')
    math_times.append(mt)
    
    speedup = mt / ft if not np.isnan(mt) else float('inf')
    print(f"seq_len={s:5d} | Flash: {ft:7.2f} ms | Math: {mt:7.2f} ms | Speedup: {speedup:.2f}x")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(seq_lens, math_times, 'ro-', linewidth=2, markersize=8, label='Math (standard)')
plt.plot(seq_lens, flash_times, 'bs-', linewidth=2, markersize=8, label='Flash Attention')
plt.xlabel('Sequence Length')
plt.ylabel('Latency (ms)')
plt.title('Attention Latency: Flash vs Standard (Math) Backend')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.savefig('latency_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Speedup chart
speedups = [m/f if not np.isnan(m) else 0 for m, f in zip(math_times, flash_times)]
plt.figure(figsize=(8, 4))
plt.bar(range(len(seq_lens)), speedups, color='#10b981')
plt.xticks(range(len(seq_lens)), seq_lens)
plt.xlabel('Sequence Length')
plt.ylabel('Speedup (x)')
plt.title('Flash Attention Speedup over Standard Attention')
plt.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('speedup_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Scaling with Sequence Length

Flash Attention's advantage grows with sequence length because:
- Standard attention: O(N²) memory + O(N²) HBM I/O
- Flash Attention: O(N) memory + O(N²/M) HBM I/O (where M = SRAM size)

The longer the sequence, the more the O(N²) HBM writes dominate in standard attention.

In [ ]:
def compute_attention_flops(seq_len, heads=32, head_dim=128):
    """FLOPs for attention: 2*N*N*d (QK^T) + 2*N*N*d (attn@V) per head."""
    return heads * (2 * seq_len * seq_len * head_dim + 2 * seq_len * seq_len * head_dim)

# Compute throughput (TFLOPS)
flash_tflops = [compute_attention_flops(s) / (t * 1e-3) / 1e12 for s, t in zip(seq_lens, flash_times)]
math_tflops = [compute_attention_flops(s) / (t * 1e-3) / 1e12 if not np.isnan(t) else 0 
               for s, t in zip(seq_lens, math_times)]

plt.figure(figsize=(10, 5))
plt.plot(seq_lens, flash_tflops, 'bs-', linewidth=2, markersize=8, label='Flash Attention')
plt.plot(seq_lens, math_tflops, 'ro-', linewidth=2, markersize=8, label='Math (standard)')
plt.axhline(y=gpu.tflops_fp16, color='green', linestyle='--', alpha=0.7, label=f'GPU Peak ({gpu.tflops_fp16} TFLOPS)')
plt.xlabel('Sequence Length')
plt.ylabel('Throughput (TFLOPS)')
plt.title('Attention Compute Throughput vs Sequence Length')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('throughput_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"
Flash Attention achieves {max(flash_tflops):.1f} TFLOPS ({max(flash_tflops)/gpu.tflops_fp16*100:.0f}% of peak)")
print(f"Standard Attention achieves {max(math_tflops):.1f} TFLOPS ({max(math_tflops)/gpu.tflops_fp16*100:.0f}% of peak)")

In [ ]:
# Demonstrate O(N^2) vs near-linear scaling
extended_seqs = [128, 256, 512, 1024, 2048, 4096]
flash_ext, math_ext = [], []

for s in extended_seqs:
    try:
        flash_ext.append(bench_sdpa(s, backend='flash', iters=30))
    except RuntimeError:
        flash_ext.append(bench_sdpa(s, backend='mem_efficient', iters=30))
    try:
        math_ext.append(bench_sdpa(s, backend='math', iters=30))
    except (torch.cuda.OutOfMemoryError, RuntimeError):
        math_ext.append(float('nan'))

# Normalize to seq_len=128 baseline
flash_norm = [t / flash_ext[0] for t in flash_ext]
math_norm = [t / math_ext[0] if not np.isnan(t) else float('nan') for t in math_ext]
seq_norm = [s / extended_seqs[0] for s in extended_seqs]
quadratic_ref = [(s / extended_seqs[0])**2 for s in extended_seqs]

plt.figure(figsize=(10, 5))
plt.loglog(extended_seqs, math_norm, 'ro-', linewidth=2, label='Standard (measured)')
plt.loglog(extended_seqs, flash_norm, 'bs-', linewidth=2, label='Flash (measured)')
plt.loglog(extended_seqs, quadratic_ref, 'k--', alpha=0.5, label='O(N²) reference')
plt.xlabel('Sequence Length')
plt.ylabel('Normalized Latency (relative to N=128)')
plt.title('Scaling Behavior: Flash Attention vs Standard')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('scaling_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Batch Size Impact on Flash Attention

Flash Attention's tiling is per-head, so it scales well with batch size. Let's measure how throughput changes with batching.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32]
seq_len = 2048
batch_flash_times = []

for b in batch_sizes:
    try:
        t = bench_sdpa(seq_len, batch=b, backend='flash')
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        t = float('nan')
    batch_flash_times.append(t)
    tput = b * seq_len / (t * 1e-3) / 1e6 if not np.isnan(t) else 0
    print(f"batch={b:3d} | Latency: {t:7.2f} ms | Throughput: {tput:.2f} M tokens/s")

# Tokens per second scaling
tokens_per_sec = [b * seq_len / (t * 1e-3) for b, t in zip(batch_sizes, batch_flash_times) if not np.isnan(t)]
valid_batches = [b for b, t in zip(batch_sizes, batch_flash_times) if not np.isnan(t)]

plt.figure(figsize=(8, 5))
plt.plot(valid_batches, [t/1e6 for t in tokens_per_sec], 'gs-', linewidth=2, markersize=8)
plt.xlabel('Batch Size')
plt.ylabel('Throughput (M tokens/s)')
plt.title(f'Flash Attention Throughput vs Batch Size (seq_len={seq_len})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('batch_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Roofline Analysis

Where does Flash Attention sit on the roofline model? Standard attention is memory-bound (low arithmetic intensity due to N² HBM traffic). Flash Attention pushes toward compute-bound by reducing HBM access.

In [ ]:
from utils.roofline import plot_roofline  # if available, else manual

# Compute arithmetic intensity for both approaches
def attention_arithmetic_intensity(seq_len, head_dim=128, heads=32, flash=False, sram_mb=20):
    """Approximate arithmetic intensity (FLOP/byte) for attention."""
    N, d, h = seq_len, head_dim, heads
    flops = h * 4 * N * N * d  # QK^T + attn@V, both are 2*N*N*d per head
    if flash:
        # Flash: reads Q,K,V once + output write = 4*N*d*h*2 bytes (simplified)
        bytes_accessed = 4 * N * d * h * 2  # approximate for well-tiled case
    else:
        # Standard: Q,K,V + S matrix + P matrix + output
        bytes_accessed = (3 * N * d * h + 2 * N * N * h + N * d * h) * 2
    return flops / bytes_accessed

seq_test = [512, 1024, 2048, 4096, 8192]
ai_standard = [attention_arithmetic_intensity(s, flash=False) for s in seq_test]
ai_flash = [attention_arithmetic_intensity(s, flash=True) for s in seq_test]

# Roofline
peak_flops = gpu.tflops_fp16 * 1e12  # FLOP/s
peak_bw = gpu.bw_gbs * 1e9  # bytes/s
ridge = gpu.ridge_point

ai_range = np.logspace(-1, 4, 100)
roofline = np.minimum(peak_flops, ai_range * peak_bw)

plt.figure(figsize=(10, 6))
plt.loglog(ai_range, roofline / 1e12, 'k-', linewidth=2, label='Roofline')
plt.axvline(x=ridge, color='gray', linestyle=':', alpha=0.5, label=f'Ridge point ({ridge:.0f} FLOP/byte)')
plt.scatter(ai_standard, [compute_attention_flops(s)/(math_times[i]*1e-3)/1e12 if i < len(math_times) and not np.isnan(math_times[i]) else 0.1 for i, s in enumerate(seq_test[:len(math_times)])],
           c='red', s=100, zorder=5, label='Standard Attention')
plt.scatter(ai_flash, [compute_attention_flops(s)/(flash_times[i]*1e-3)/1e12 if i < len(flash_times) else 0.1 for i, s in enumerate(seq_test[:len(flash_times)])],
           c='blue', s=100, zorder=5, label='Flash Attention')
plt.xlabel('Arithmetic Intensity (FLOP/byte)')
plt.ylabel('Performance (TFLOPS)')
plt.title('Roofline Model: Standard vs Flash Attention')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roofline_attention.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Standard attention AI at N=4096: {ai_standard[3]:.1f} FLOP/byte (memory-bound)")
print(f"Flash attention AI at N=4096: {ai_flash[3]:.1f} FLOP/byte (closer to compute-bound)")

## 8. Key Findings

### Performance
- Flash Attention provides **2-10x speedup** over standard attention, growing with sequence length
- The speedup comes from **reduced HBM I/O**, not more compute — same FLOPs, fewer memory transfers
- Flash Attention achieves higher GPU utilization (closer to peak TFLOPS)

### Memory
- Standard attention: O(N²) peak memory — **limits max sequence length**
- Flash Attention: O(N) peak memory — enables much longer contexts
- At N=4096 with 32 heads: standard needs ~2 GB for attention matrices; flash needs ~0

### Scaling
- Standard attention latency scales **quadratically** with sequence length
- Flash Attention scales **sub-quadratically** in practice (same O(N²) FLOPs but constant-factor better due to tiling)
- The gap widens at longer sequences — flash is essential for 8K+ contexts

### Practical implications for LLM inference
- **Prefill phase**: Flash Attention is critical — processing long prompts hits the N² wall fast
- **Decode phase**: Less impactful (N=1 for the new token, but KV cache attention still benefits)
- **All modern serving engines** (vLLM, TensorRT-LLM, TGI) use Flash Attention by default
- PyTorch's  auto-selects flash backend when available

In [ ]:
# Cleanup
torch.cuda.empty_cache()
print("Notebook complete. Generated plots saved to current directory.")
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1024**2:.1f} MB still allocated")